# 06 — Autograd and Automatic Differentiation

In the previous notebook, we learned how matrix multiplication powers linear transformations and neural-network layers.

Now we will study the mechanism that allows neural networks to **learn**:

> **Automatic differentiation**

PyTorch's automatic differentiation system is called **Autograd**.

Autograd tracks operations performed on tensors and automatically computes derivatives and gradients.

## In this notebook, we will learn:

1. Derivatives from zero
2. Gradients
3. `requires_grad`
4. Computational graphs
5. `grad_fn`
6. `backward()`
7. Gradient accumulation
8. Clearing gradients
9. `torch.no_grad()`
10. `detach()`
11. Chain rule intuition
12. Manual gradients vs PyTorch gradients
13. Gradients for multiple variables
14. Gradients for vectors
15. Why gradients matter in neural networks
16. Common autograd mistakes
17. Debugging gradients
18. Practice exercises

## Main Goal

By the end of this notebook, you should understand the complete idea:

$$
\text{Forward Pass}
\rightarrow
\text{Loss}
\rightarrow
\text{Backward Pass}
\rightarrow
\text{Gradients}
\rightarrow
\text{Parameter Update}
$$

The most important question to ask is:

> **How does changing a parameter change the final output or loss?**

That sensitivity is what a gradient measures.


In [ ]:
import torch

print("PyTorch version:", torch.__version__)


# 1. Derivatives From Zero

Before using Autograd, we need the basic idea of a **derivative**.

A derivative measures:

> **How much does an output change when an input changes slightly?**

Consider:

$$
y=x^2
$$

If:

$$
x=3
$$

then:

$$
y=3^2=9
$$

The derivative is:

$$
\frac{dy}{dx}=2x
$$

At:

$$
x=3
$$

we get:

$$
\frac{dy}{dx}=2(3)=\boxed{6}
$$

This means that near $x=3$, a small increase in $x$ causes $y$ to increase at a rate of approximately `6` units of output per unit of input.


## Numerical Intuition

Suppose:

$$
f(x)=x^2
$$

At:

$$
x=3
$$

we can estimate the slope using a very small change:

$$
\frac{f(x+h)-f(x)}{h}
$$

where $h$ is small.


In [ ]:
def f(x):
    return x ** 2

x = 3.0
h = 0.0001

numerical_derivative = (f(x + h) - f(x)) / h

print("Numerical derivative:", numerical_derivative)
print("Exact derivative:", 2 * x)


The numerical answer should be very close to:

$$
\boxed{6}
$$

The smaller $h$ becomes, the closer the approximation usually gets until numerical precision becomes a concern.


# 2. What Is a Gradient?

A **derivative** usually refers to the rate of change with respect to one variable.

A **gradient** extends this idea to functions with multiple inputs.

Suppose:

$$
f(x,y)=x^2+y^2
$$

Then:

$$
\frac{\partial f}{\partial x}=2x
$$

and:

$$
\frac{\partial f}{\partial y}=2y
$$

The gradient is:

$$
\nabla f=
\begin{bmatrix}
\frac{\partial f}{\partial x} \\
\frac{\partial f}{\partial y}
\end{bmatrix}
$$

At:

$$
x=3,\qquad y=4
$$

the gradient is:

$$
\nabla f=
\begin{bmatrix}
6 \\
8
\end{bmatrix}
$$

In deep learning, model parameters may contain millions or billions of values.

Autograd computes gradients for these parameters automatically.


# 3. `requires_grad`

By default, PyTorch does **not** track gradients for every tensor.

To tell PyTorch that a tensor should participate in gradient computation, use:

`requires_grad=True`


In [ ]:
x = torch.tensor(3.0, requires_grad=True)

print(x)
print("requires_grad:", x.requires_grad)


Now PyTorch knows:

> Operations involving `x` may later need derivatives with respect to `x`.

This is especially important for model parameters such as:

- Weights
- Biases


# 4. First Autograd Example

Let:

$$
y=x^2
$$

and:

$$
x=3
$$

Mathematically:

$$
\frac{dy}{dx}=2x
$$

Therefore:

$$
\frac{dy}{dx}\Big|_{x=3}=6
$$

Let's ask PyTorch to compute it.


In [ ]:
x = torch.tensor(3.0, requires_grad=True)

y = x ** 2

print("x:", x)
print("y:", y)


To compute the gradient, call:

`y.backward()`


In [ ]:
y.backward()

print("x.grad:", x.grad)


PyTorch returns:

$$
\boxed{6}
$$

because:

$$
\frac{d(x^2)}{dx}=2x
$$

and:

$$
2(3)=6
$$


# 5. Understanding `.grad`

After `backward()` runs, the gradient is stored in:

`tensor.grad`

For the previous example:

`x.grad`

contains:

$$
\frac{dy}{dx}
$$

evaluated at the current value of `x`.


In [ ]:
x = torch.tensor(5.0, requires_grad=True)

y = x ** 2
y.backward()

print("x:", x.item())
print("y:", y.item())
print("dy/dx:", x.grad.item())


Since:

$$
\frac{dy}{dx}=2x
$$

at:

$$
x=5
$$

the gradient should be:

$$
\boxed{10}
$$


# 6. Computational Graphs

Autograd works by building a **computational graph**.

Consider:

$$
x=2
$$

$$
a=x^2
$$

$$
b=3a
$$

$$
y=b+1
$$

Conceptually:

$$
x
\rightarrow
x^2
\rightarrow
3x^2
\rightarrow
3x^2+1
$$

Every operation becomes part of a graph.

During the backward pass, PyTorch walks backward through this graph and applies the chain rule.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

a = x ** 2
b = 3 * a
y = b + 1

print("x =", x)
print("a =", a)
print("b =", b)
print("y =", y)


# 7. `grad_fn`

Tensors created by operations involving tracked tensors usually contain a:

`grad_fn`

This tells us which operation created the tensor.


In [ ]:
print("x.grad_fn:", x.grad_fn)
print("a.grad_fn:", a.grad_fn)
print("b.grad_fn:", b.grad_fn)
print("y.grad_fn:", y.grad_fn)


Notice:

- `x` is a tensor we created directly, so `x.grad_fn` is usually `None`
- `a`, `b`, and `y` were produced by operations, so they have gradient functions

This is evidence that PyTorch is building a computational graph.


# 8. Leaf Tensors

A tensor like:

`x = torch.tensor(2.0, requires_grad=True)`

is usually a **leaf tensor**.

Leaf tensors are important because gradients are normally accumulated in `.grad` for leaf tensors.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2

print("x.is_leaf:", x.is_leaf)
print("y.is_leaf:", y.is_leaf)


Model weights and biases are typically leaf tensors that require gradients.

That is why their gradients are available after backpropagation.


# 9. `backward()`

`backward()` starts the backward pass.

Consider:

$$
y=3x^2+1
$$

Then:

$$
\frac{dy}{dx}=6x
$$

At:

$$
x=2
$$

the gradient is:

$$
\boxed{12}
$$


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y = 3 * x ** 2 + 1

y.backward()

print("y:", y.item())
print("x.grad:", x.grad.item())


# 10. Chain Rule Intuition

The **chain rule** is the mathematical idea that makes backpropagation possible.

Suppose:

$$
u=x^2
$$

and:

$$
y=3u+1
$$

We want:

$$
\frac{dy}{dx}
$$

The chain rule says:

$$
\frac{dy}{dx}
=
\frac{dy}{du}
\cdot
\frac{du}{dx}
$$

Now:

$$
\frac{dy}{du}=3
$$

and:

$$
\frac{du}{dx}=2x
$$

Therefore:

$$
\frac{dy}{dx}=3(2x)=6x
$$

At $x=2$:

$$
\boxed{12}
$$

Autograd performs this chain-rule calculation automatically through the computational graph.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

u = x ** 2
y = 3 * u + 1

y.backward()

print("Autograd result:", x.grad)


# 11. Multiple Operations

Consider:

$$
y=(x+2)^3
$$

Using the chain rule:

$$
\frac{dy}{dx}
=
3(x+2)^2
$$

At:

$$
x=1
$$

we get:

$$
3(3)^2=27
$$


In [ ]:
x = torch.tensor(1.0, requires_grad=True)

y = (x + 2) ** 3

y.backward()

print("Gradient:", x.grad)


# 12. Gradients With Multiple Variables

Suppose:

$$
z=x^2+3y
$$

Then:

$$
\frac{\partial z}{\partial x}=2x
$$

and:

$$
\frac{\partial z}{\partial y}=3
$$

At:

$$
x=2,\qquad y=4
$$

we expect:

$$
\frac{\partial z}{\partial x}=4
$$

$$
\frac{\partial z}{\partial y}=3
$$


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(4.0, requires_grad=True)

z = x ** 2 + 3 * y

z.backward()

print("dz/dx:", x.grad)
print("dz/dy:", y.grad)


# 13. A Two-Variable Example

Consider:

$$
z=xy+x^2
$$

Then:

$$
\frac{\partial z}{\partial x}=y+2x
$$

and:

$$
\frac{\partial z}{\partial y}=x
$$

At:

$$
x=2,\qquad y=3
$$

we expect:

$$
\frac{\partial z}{\partial x}=3+4=7
$$

$$
\frac{\partial z}{\partial y}=2
$$


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

z = x * y + x ** 2

z.backward()

print("dz/dx:", x.grad)
print("dz/dy:", y.grad)


# 14. Gradient Accumulation

This is one of the most important PyTorch behaviors to understand:

> **Gradients accumulate by default.**

Suppose we call `backward()` multiple times on new graphs while reusing the same leaf tensor.

The new gradient is **added** to the existing `.grad`.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y1 = x ** 2
y1.backward()

print("After first backward:", x.grad)

y2 = x ** 2
y2.backward()

print("After second backward:", x.grad)


For:

$$
y=x^2
$$

at:

$$
x=2
$$

each backward pass contributes:

$$
\frac{dy}{dx}=4
$$

So after two backward passes:

$$
4+4=\boxed{8}
$$

This accumulation is intentional.

During neural-network training, we normally clear gradients before computing gradients for the next optimization step.


# 15. Clearing Gradients

For an individual tensor, you can clear the gradient using:

`x.grad.zero_()`


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2
y.backward()

print("Before clearing:", x.grad)

x.grad.zero_()

print("After clearing:", x.grad)


Another option is:

`x.grad = None`

PyTorch optimizers commonly use:

`optimizer.zero_grad()`

We will study optimizers later.

For now, remember:

> **If you do not clear gradients, they accumulate.**


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2
y.backward()

print("Before:", x.grad)

x.grad = None

print("After setting to None:", x.grad)


# 16. Why Calling `backward()` Twice on the Same Graph Can Fail

By default, PyTorch frees intermediate graph information after a backward pass.

So this may fail:

```python
y.backward()
y.backward()
```

because the same graph is being reused.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3

y.backward()

print("First gradient:", x.grad)

# Uncomment the next line to see the error:
# y.backward()


If you genuinely need to reuse the same graph, you can use:

`retain_graph=True`

This is not usually needed in ordinary training loops.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3

y.backward(retain_graph=True)
print("After first backward:", x.grad)

y.backward()
print("After second backward:", x.grad)


# 17. `torch.no_grad()`

Sometimes we want to perform computations **without tracking gradients**.

This is common during:

- Validation
- Testing
- Inference
- Parameter updates performed manually

Use:

`with torch.no_grad():`


In [ ]:
x = torch.tensor(3.0, requires_grad=True)

with torch.no_grad():
    y = x ** 2

print("y:", y)
print("y.requires_grad:", y.requires_grad)
print("y.grad_fn:", y.grad_fn)


Compare this with normal gradient tracking.


In [ ]:
x = torch.tensor(3.0, requires_grad=True)

y = x ** 2

print("y.requires_grad:", y.requires_grad)
print("y.grad_fn:", y.grad_fn)


# 18. Why `torch.no_grad()` Matters

During model inference, we only need predictions.

We do not need:

- Computational graphs
- Backward passes
- Gradient storage

Disabling gradient tracking can reduce memory usage and unnecessary computation.

Later, you will often see:

```python
model.eval()

with torch.no_grad():
    predictions = model(x)
```


# 19. `detach()`

`detach()` returns a tensor that shares the same underlying data but is disconnected from the current computational graph.


In [ ]:
x = torch.tensor(3.0, requires_grad=True)

y = x ** 2

y_detached = y.detach()

print("y:", y)
print("y.requires_grad:", y.requires_grad)
print("y_detached:", y_detached)
print("y_detached.requires_grad:", y_detached.requires_grad)


Conceptually:

$$
x
\rightarrow
y=x^2
$$

is tracked.

But after:

`y.detach()`

the detached tensor no longer tracks the history that produced it.

This is useful when you need tensor values but do not want future operations to backpropagate through the earlier graph.


# 20. `no_grad()` vs `detach()`

$$
\begin{array}{|c|c|}
\hline
\textbf{torch.no_grad()} & \textbf{detach()} \\
\hline
\text{Temporarily disables tracking in a block} & \text{Disconnects a particular tensor} \\
\hline
\text{Useful for inference} & \text{Useful for stopping gradient flow} \\
\hline
\text{Affects operations inside the context} & \text{Returns a detached tensor} \\
\hline
\end{array}
$$


# 21. Manual Gradient vs PyTorch Gradient

Let's compare mathematics and Autograd directly.

Consider:

$$
y=x^3+2x^2+5x+1
$$

The derivative is:

$$
\frac{dy}{dx}
=
3x^2+4x+5
$$

At:

$$
x=2
$$

the manual gradient is:

$$
3(2)^2+4(2)+5
$$

$$
=12+8+5
$$

$$
=\boxed{25}
$$


In [ ]:
x_value = 2.0

manual_gradient = 3 * x_value ** 2 + 4 * x_value + 5

print("Manual gradient:", manual_gradient)


In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 3 + 2 * x ** 2 + 5 * x + 1

y.backward()

print("PyTorch gradient:", x.grad.item())


The two answers should match.

This is the central benefit of Autograd:

> We define the forward computation, and PyTorch automatically derives the backward computation.


# 22. Vector Gradients

Suppose:

$$
x=
\begin{array}{|c|c|c|}
\hline
1 & 2 & 3 \\
\hline
\end{array}
$$

and:

$$
y=\sum_i x_i^2
$$

Then:

$$
y=1^2+2^2+3^2=14
$$

The gradient is:

$$
\frac{\partial y}{\partial x}
=
\begin{array}{|c|c|c|}
\hline
2 & 4 & 6 \\
\hline
\end{array}
$$


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

y = (x ** 2).sum()

y.backward()

print("y:", y)
print("x.grad:", x.grad)


# 23. Why We Often Reduce to a Scalar Before `backward()`

For a scalar output:

$$
y \in \mathbb{R}
$$

calling:

`y.backward()`

is straightforward.

But suppose:

$$
y=x^2
$$

where `x` is a vector.

Then `y` is also a vector.

There is no single scalar gradient unless we specify how the vector output should be combined.


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

y = x ** 2

print("y:", y)
print("y shape:", y.shape)

# Uncomment to see the error:
# y.backward()


A common solution is to reduce the vector to a scalar:

`y.sum().backward()`


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

y = x ** 2

y.sum().backward()

print("x.grad:", x.grad)


# 24. Backward With an Explicit Gradient

For non-scalar outputs, PyTorch also allows you to provide a gradient argument.

Suppose:

$$
y=x^2
$$

with:

$$
x=
\begin{array}{|c|c|c|}
\hline
1 & 2 & 3 \\
\hline
\end{array}
$$

If we call:

`y.backward(gradient=v)`

PyTorch computes a vector-Jacobian product using `v`.

For now, the key idea is:

> Non-scalar outputs need additional information for `backward()`.


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

y = x ** 2

v = torch.tensor([1.0, 1.0, 1.0])

y.backward(gradient=v)

print("x.grad:", x.grad)


# 25. Gradients Through Matrix Operations

Autograd works through matrix multiplication too.

Suppose:

$$
XW
$$

is part of a model.

PyTorch can compute gradients with respect to:

- `X`
- `W`
- Biases
- Any other tracked parameter


In [ ]:
X = torch.tensor([
    [1.0, 2.0],
    [3.0, 4.0]
])

W = torch.tensor([
    [0.5, 1.0],
    [-1.0, 2.0]
], requires_grad=True)

Y = X @ W

loss = Y.sum()

loss.backward()

print("Y:")
print(Y)

print("\\nLoss:", loss)
print("\\nGradient of W:")
print(W.grad)


This is exactly the type of computation that happens inside neural networks.

The forward pass uses matrix multiplication.

The backward pass computes gradients through those matrix operations.


# 26. A Single Neuron With Autograd

A neuron can be written as:

$$
z=x_1w_1+x_2w_2+x_3w_3+b
$$

Suppose the target is:

$$
t=10
$$

and our loss is:

$$
L=(z-t)^2
$$

Autograd can compute:

$$
\frac{\partial L}{\partial w_1},
\frac{\partial L}{\partial w_2},
\frac{\partial L}{\partial w_3},
\frac{\partial L}{\partial b}
$$

automatically.


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])

w = torch.tensor([0.5, -1.0, 2.0], requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

target = torch.tensor(10.0)

prediction = torch.dot(x, w) + b
loss = (prediction - target) ** 2

loss.backward()

print("Prediction:", prediction.item())
print("Loss:", loss.item())
print("w.grad:", w.grad)
print("b.grad:", b.grad)


# 27. Why Gradients Matter in Neural Networks

Training a neural network means finding parameter values that reduce the loss.

The basic process is:

$$
\text{Parameters}
\rightarrow
\text{Prediction}
\rightarrow
\text{Loss}
\rightarrow
\text{Gradient}
\rightarrow
\text{Updated Parameters}
$$

The gradient tells us how the loss changes with respect to each parameter.

If:

$$
\frac{\partial L}{\partial w}>0
$$

increasing $w$ increases the loss locally.

So gradient descent moves in the opposite direction:

$$
w_{new}
=
w_{old}
-
\eta
\frac{\partial L}{\partial w}
$$

where:

$$
\eta
$$

is the learning rate.


# 28. One Manual Gradient-Descent Step

Let's use one parameter:

$$
y=wx
$$

Target:

$$
t=8
$$

Loss:

$$
L=(wx-t)^2
$$

We will:

1. Compute the prediction
2. Compute the loss
3. Call `backward()`
4. Update `w`


In [ ]:
x = torch.tensor(2.0)
target = torch.tensor(8.0)

w = torch.tensor(1.0, requires_grad=True)

prediction = w * x
loss = (prediction - target) ** 2

loss.backward()

print("Prediction before update:", prediction.item())
print("Loss before update:", loss.item())
print("Gradient:", w.grad.item())


Now update the weight in the **opposite direction of the gradient**.

We use `torch.no_grad()` because the parameter update itself should not become part of the computational graph.


In [ ]:
learning_rate = 0.1

with torch.no_grad():
    w -= learning_rate * w.grad

print("Updated w:", w.item())


Now clear the gradient before the next step.


In [ ]:
w.grad.zero_()

print("Cleared gradient:", w.grad)


# 29. Repeating Gradient Descent

Now we can repeat the process several times.

This is the beginning of a real training loop.


In [ ]:
x = torch.tensor(2.0)
target = torch.tensor(8.0)

w = torch.tensor(1.0, requires_grad=True)

learning_rate = 0.1

for step in range(10):
    prediction = w * x
    loss = (prediction - target) ** 2

    loss.backward()

    with torch.no_grad():
        w -= learning_rate * w.grad

    w.grad.zero_()

    print(
        f"Step {step + 1:02d} | "
        f"w = {w.item():.4f} | "
        f"loss = {loss.item():.4f}"
    )


The ideal weight is:

$$
w=4
$$

because:

$$
4\times2=8
$$

You should see `w` move toward `4` and the loss decrease.

This small example contains the core logic behind neural-network training.


# 30. Inspecting Gradient Tracking

Useful properties include:

- `requires_grad`
- `grad`
- `grad_fn`
- `is_leaf`


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
z = y + 5

print("x.requires_grad:", x.requires_grad)
print("x.is_leaf:", x.is_leaf)
print("x.grad_fn:", x.grad_fn)

print()

print("y.requires_grad:", y.requires_grad)
print("y.is_leaf:", y.is_leaf)
print("y.grad_fn:", y.grad_fn)

print()

print("z.requires_grad:", z.requires_grad)
print("z.is_leaf:", z.is_leaf)
print("z.grad_fn:", z.grad_fn)


# 31. Common Autograd Mistakes

## Mistake 1 — Forgetting `requires_grad=True`

If PyTorch is not tracking a tensor, it cannot automatically compute its gradient.

## Mistake 2 — Forgetting That Gradients Accumulate

Repeated backward passes add to `.grad`.

Clear gradients between optimization steps.

## Mistake 3 — Calling `backward()` on a Non-Scalar Without a Gradient

A non-scalar output needs:

- A reduction such as `.sum()`, or
- An explicit gradient argument

## Mistake 4 — Updating Parameters While Tracking Gradients

Manual parameter updates should normally happen inside:

`with torch.no_grad():`

## Mistake 5 — Expecting `.grad` on Every Intermediate Tensor

Gradients are normally retained automatically for leaf tensors, not every intermediate tensor.

## Mistake 6 — Calling `backward()` Twice on the Same Freed Graph

If the same graph must be reused, `retain_graph=True` may be required.

## Mistake 7 — Using `detach()` Too Early

Detaching a tensor stops gradient flow through the previous graph.

Use it intentionally.


# 32. Autograd Debugging Checklist

When gradients are missing or incorrect, inspect:

- `tensor.requires_grad`
- `tensor.grad`
- `tensor.grad_fn`
- `tensor.is_leaf`
- Tensor shapes
- Tensor dtypes
- Whether `detach()` was used
- Whether computation occurred inside `torch.no_grad()`

Then ask:

1. Is this tensor supposed to receive gradients?
2. Is it a leaf tensor?
3. Did I call `backward()`?
4. Is the output scalar?
5. Did I accidentally detach the tensor?
6. Did I clear gradients at the correct time?
7. Am I updating parameters inside `torch.no_grad()`?


In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2

print("Before backward:")
print("x.grad:", x.grad)
print("y.grad_fn:", y.grad_fn)

y.backward()

print("\\nAfter backward:")
print("x.grad:", x.grad)


# 33. Practice Exercises

Try solving these without looking at the solutions.

## Exercise 1

For:

$$
y=x^2
$$

calculate the derivative manually at:

$$
x=4
$$

Then verify it with Autograd.

## Exercise 2

For:

$$
y=5x^3
$$

find:

$$
\frac{dy}{dx}
$$

at:

$$
x=2
$$

using PyTorch.

## Exercise 3

For:

$$
z=x^2+y^2
$$

at:

$$
x=3,\qquad y=4
$$

find:

- `dz/dx`
- `dz/dy`

## Exercise 4

Create:

`x = torch.tensor(2.0, requires_grad=True)`

Compute:

$$
y=4x^2+3x+1
$$

and find `x.grad`.

## Exercise 5

Demonstrate gradient accumulation by calling backward on two newly constructed expressions using the same leaf tensor.

## Exercise 6

Clear a tensor's gradient using:

`x.grad.zero_()`

## Exercise 7

Use `torch.no_grad()` to compute:

$$
y=x^2
$$

from a tensor where `x.requires_grad=True`.

Check whether `y.requires_grad` is `True` or `False`.

## Exercise 8

Use `detach()` and verify that the detached tensor does not require gradients.

## Exercise 9

For:

$$
x=
\begin{array}{|c|c|c|}
\hline
1 & 2 & 3 \\
\hline
\end{array}
$$

compute:

$$
y=\sum x_i^2
$$

and find the gradient with respect to `x`.

## Exercise 10

Create a single-neuron squared-error loss and compute gradients for its weights and bias.


# 34. Conceptual Challenges

Answer these without running code first.

## Challenge 1

If:

$$
y=x^3
$$

what should `x.grad` be at:

$$
x=2
$$

after calling `y.backward()`?

## Challenge 2

Why does PyTorch accumulate gradients instead of automatically replacing them?

## Challenge 3

Why are manual parameter updates commonly placed inside:

`torch.no_grad()`?

## Challenge 4

What happens to gradient flow after:

`y = x.detach()`?

## Challenge 5

Why might:

`y.backward()`

fail if `y` contains multiple elements?

## Challenge 6

What is the relationship between:

- Computational graphs
- Chain rule
- Backpropagation
- Autograd

## Challenge 7

Why do neural-network weights usually have:

`requires_grad=True`

while input labels usually do not?


# 35. Exercise Solutions


In [ ]:
# Exercise 1
x = torch.tensor(4.0, requires_grad=True)
y = x ** 2
y.backward()
print("Exercise 1:", x.grad)  # 8

# Exercise 2
x = torch.tensor(2.0, requires_grad=True)
y = 5 * x ** 3
y.backward()
print("Exercise 2:", x.grad)  # 60

# Exercise 3
x = torch.tensor(3.0, requires_grad=True)
y = torch.tensor(4.0, requires_grad=True)
z = x ** 2 + y ** 2
z.backward()
print("Exercise 3 dz/dx:", x.grad)
print("Exercise 3 dz/dy:", y.grad)

# Exercise 4
x = torch.tensor(2.0, requires_grad=True)
y = 4 * x ** 2 + 3 * x + 1
y.backward()
print("Exercise 4:", x.grad)

# Exercise 5
x = torch.tensor(2.0, requires_grad=True)
(x ** 2).backward()
(x ** 2).backward()
print("Exercise 5 accumulated:", x.grad)

# Exercise 6
x.grad.zero_()
print("Exercise 6 cleared:", x.grad)

# Exercise 7
x = torch.tensor(2.0, requires_grad=True)
with torch.no_grad():
    y = x ** 2
print("Exercise 7 requires_grad:", y.requires_grad)

# Exercise 8
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
z = y.detach()
print("Exercise 8 requires_grad:", z.requires_grad)

# Exercise 9
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = (x ** 2).sum()
y.backward()
print("Exercise 9:", x.grad)

# Exercise 10
features = torch.tensor([1.0, 2.0, 3.0])
weights = torch.tensor([0.5, 0.5, 0.5], requires_grad=True)
bias = torch.tensor(0.0, requires_grad=True)
target = torch.tensor(5.0)

prediction = torch.dot(features, weights) + bias
loss = (prediction - target) ** 2
loss.backward()

print("Exercise 10 weights grad:", weights.grad)
print("Exercise 10 bias grad:", bias.grad)


# 36. Key Takeaways

In this notebook, we learned:

- What derivatives mean
- What gradients mean
- `requires_grad`
- Computational graphs
- `grad_fn`
- Leaf tensors
- `backward()`
- Chain rule intuition
- Gradient accumulation
- Clearing gradients
- `torch.no_grad()`
- `detach()`
- Manual gradients vs Autograd
- Vector gradients
- Matrix-operation gradients
- A neuron with Autograd
- Manual gradient descent

The central idea is:

$$
\text{Forward computation}
\rightarrow
\text{Loss}
\rightarrow
\text{Backward computation}
\rightarrow
\text{Gradients}
$$

Those gradients tell us how to change model parameters to reduce the loss.


# 37. Check Your Understanding

Before moving to the next notebook, make sure you can answer these without searching:

1. What does a derivative measure?
2. What is a gradient?
3. What does `requires_grad=True` do?
4. What is a computational graph?
5. What does `grad_fn` tell us?
6. What is a leaf tensor?
7. What does `backward()` do?
8. Where are gradients stored?
9. Why do gradients accumulate?
10. How do you clear gradients?
11. What does `torch.no_grad()` do?
12. What does `detach()` do?
13. What is the chain rule?
14. Why do non-scalar outputs need special handling with `backward()`?
15. Why are gradients essential for neural-network training?
16. Why are parameter updates performed in the opposite direction of the gradient?
17. Why should manual parameter updates usually happen without gradient tracking?


# Next Notebook

# 07 — Linear Regression From Scratch

In the next notebook, we will study:

- Linear regression intuition
- The equation $y=wx+b$
- Synthetic data
- Prediction / forward pass
- Mean squared error
- Manual gradients
- Autograd gradients
- Gradient descent
- Training loop
- Learning rate
- Loss curves
- Comparing manual training with `nn.Linear`
- Common training mistakes
